# Solution B3 · Skewed variables: log, Yeo-Johnson and quantiles, inside and outside the range

**Type**: Implement with scikit-learn

## Objective

Compare three transformations of skewed variables (log, `PowerTransformer`, `QuantileTransformer`) on the skewness and
on the performance of a linear model, and verify that the `QuantileTransformer` **cannot extrapolate**.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, PowerTransformer, QuantileTransformer

rng = np.random.default_rng(0)
n = 1200
X = pd.DataFrame({"x1": rng.lognormal(0, 1, n), "x2": rng.lognormal(1, 0.8, n), "x3": rng.normal(size=n)})
y = 2 * np.log(X.x1) - 1.5 * np.log(X.x2) + X.x3 + rng.normal(0, 0.5, n)
X_tr, X_te = X.iloc[:600], X.iloc[600:]
y_tr, y_te = y.iloc[:600], y.iloc[600:]
X_extreme = X_te.copy()
X_extreme[["x1", "x2"]] *= 20                                   # outside the training range
y_extreme = 2 * np.log(X_extreme.x1) - 1.5 * np.log(X_extreme.x2) + X_extreme.x3 + (y_te - (2 * np.log(X_te.x1) - 1.5 * np.log(X_te.x2) + X_te.x3))
COL = ["x1", "x2"]

## Commented solution

In [ ]:
transforms = {
    "raw": "passthrough",
    "log": FunctionTransformer(np.log),
    "yeo_johnson": PowerTransformer(),
    "quantile": QuantileTransformer(n_quantiles=200, output_distribution="normal"),
}
skewness = pd.DataFrame(index=COL, columns=list(transforms), dtype=float)
for name, t in transforms.items():
    for c in COL:
        z = X_tr[[c]].to_numpy() if isinstance(t, str) else t.fit_transform(X_tr[[c]])
        skewness.loc[c, name] = stats.skew(np.asarray(z).ravel())
skewness.round(2)

In [ ]:
r2_normal, r2_extreme = {}, {}
for name, t in transforms.items():
    model = make_pipeline(ColumnTransformer([("t", t, COL)], remainder="passthrough"), Ridge(alpha=1)).fit(X_tr, y_tr)
    r2_normal[name] = r2_score(y_te, model.predict(X_te))
    r2_extreme[name] = r2_score(y_extreme, model.predict(X_extreme))
best_normal = max(r2_normal, key=r2_normal.get)
best_extreme = max(r2_extreme, key=r2_extreme.get)
print("R2 on the normal test set :", {k: round(v, 3) for k, v in r2_normal.items()})
print("R2 on the extreme test set:", {k: round(v, 3) for k, v in r2_extreme.items()})

In [ ]:
answer = "a"

## Check

In [ ]:
with verify():
    check(list(skewness.columns) == ["raw", "log", "yeo_johnson", "quantile"], "correct columns", "raw, log, yeo_johnson, quantile")
    check((skewness["raw"] > 2).all() and (skewness.drop(columns="raw").abs() < 0.3).all().all(),
          "the three transformations reduce the skewness (from above 2 to below 0.3 in absolute value)", "the skewness of the transformed ones must be close to zero")
    check(r2_normal["log"] > r2_normal["raw"] + 0.2, "the log improves the linear model a lot compared to the raw",
          "r2 with the log must exceed the raw one by more than 0.2")
    check(abs(r2_normal["yeo_johnson"] - r2_normal["log"]) < 0.05, "Yeo-Johnson is equivalent to the log on the normal data", "on the data inside the range they must almost coincide")
    check(r2_extreme["log"] > r2_extreme["quantile"] + 0.3, "out of range the log stays good and the quantiles collapse",
          "on the extreme data the quantile transformer should be much worse than the log")
    check(best_extreme in ("log", "yeo_johnson") and best_extreme != "quantile", "the best out of range is not the quantile transformer", "look at r2_extreme")
    check(answer == "a", "correct explanation", "reread: what does QuantileTransformer do with a value beyond the maximum seen?")
done()

## What you learned

- Log and Yeo-Johnson bring the skewness close to zero and let the linear model find the true relationship again
  (log): R² goes up from about 0.71 to about 0.96 compared to the raw features.
- The `QuantileTransformer` produces perfect Gaussian distributions but it is a step function computed on the training set:
  out of range it **saturates**. A customer with an income 20 times the maximum seen in training is indistinguishable from the maximum.
  It is a real limit of all the transformations based on quantiles. Out of range the log stays at 0.96 because it coincides with the true relationship, Yeo-Johnson drops to about 0.56 (its power approximates the log only inside the range) and the quantiles to about 0.46.
- Yeo-Johnson generalizes the log (it also accepts zero and negative values) and estimates the power by itself: it is a good default choice
  for skewed features, but it remains an approximation: if you know the true relationship (here the log), using it beats the automatic estimate.

## Common pitfall

Choosing the transformation by looking only at the skewness (`skew` close to zero) or only at the in-range test set. Neither
tells you what happens when the production data leave the range: it is a common case for incomes, prices and counts.

## Interview question

> *Click on a question to show its answer.*

<details><summary>You have a feature with a long tail. Log or QuantileTransformer?</summary>

I start from the model: for trees and boosting nothing is needed (monotonic transformations do not change the splits). For a linear
or distance-based model I prefer the log or Yeo-Johnson: they are continuous, monotonic functions that keep
working out of range and keep an interpretation (percentage changes). I use the quantile transformer when the
distribution is very irregular (bimodal) and I don't expect values outside the range seen.

</details>